# Exploratory Data Analysis: FIFA Player Rating Pipeline

This notebook records the data checks used to design the cleaning and feature-engineering steps in the Kedro project. It uses the raw player-attribute sample and the shared utility functions from `src/mlops_player_rating/` so that the exploratory checks stay close to the pipeline implementation.

## 1. Load and normalize the raw table

Input: `data/01_raw/player_attributes.csv`, a deterministic 6,000-row extract from the European Soccer Database player-attribute snapshots. Column normalization is applied before profiling so the notebook uses the same names as the project pipelines.


In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from mlops_player_rating import utils

raw = pd.read_csv('../data/01_raw/player_attributes.csv')
df = utils.normalize_columns(raw)
print(df.shape)
df.head()

## 2. Target distribution

`overall_rating` ranges from 42 to 90 in this sample, with a mean close to 68 and no heavy right tail. The target is already on an interpretable 0-100 rating scale, so the model can predict rating points directly instead of using a transformed target.


In [ ]:
print(df['overall_rating'].describe().round(2))
df['overall_rating'].hist(bins=40); plt.title('overall_rating'); plt.xlabel('rating'); plt.show()

## 3. Data quality checks used by the pipeline

The next cell profiles the raw sample and builds the cleaning summary from the observed values. The implementation choices match the cleaning and feature-engineering code used by the Kedro pipeline.


In [ ]:
# Dirty work-rate categories and missingness summary
work_rate_cols = ['attacking_work_rate', 'defensive_work_rate']
valid_work_rates = {'low', 'medium', 'high'}

invalid_work_rates = {}
for col in work_rate_cols:
    values = set(df[col].dropna().astype(str).str.lower().unique())
    invalid_work_rates[col] = sorted(values - valid_work_rates)

null_target_rows = int(df['overall_rating'].isna().sum())
attribute_nulls = df[utils.SKILL_COLUMNS].isna().sum().sort_values(ascending=False)
attribute_cols_with_nulls = int((attribute_nulls > 0).sum())
max_attribute_nulls = int(attribute_nulls.max())
potential_corr = round(df[['potential', 'overall_rating']].corr().iloc[0, 1], 3)

print('attacking_work_rate raw values:')
print(df['attacking_work_rate'].value_counts(dropna=False).head(12))
print()
print('null target rows:', null_target_rows)
print()
print('top-null attribute columns:')
print(attribute_nulls.head(8))

pd.DataFrame([
    {
        'check': 'missing target',
        'observed': f'{null_target_rows} rows',
        'pipeline handling': 'drop before model training and evaluation',
    },
    {
        'check': 'work-rate values',
        'observed': '; '.join(f'{col}: {vals}' for col, vals in invalid_work_rates.items() if vals),
        'pipeline handling': 'map valid labels; replace invalid or missing labels with medium',
    },
    {
        'check': 'attribute missingness',
        'observed': f'{attribute_cols_with_nulls} skill columns with nulls; max {max_attribute_nulls} rows in one column',
        'pipeline handling': 'use saved medians fitted in the cleaning step for training and inference',
    },
    {
        'check': 'potential field',
        'observed': f'corr with overall_rating = {potential_corr}',
        'pipeline handling': 'exclude from model features',
    },
])


## 4. Feature signal and engineered columns

The correlation table keeps `potential` visible only to document the exclusion decision. Among retained inputs, `reactions`, passing, ball-control, shooting, and vision features carry the strongest individual linear signal.

The feature-engineering step adds `age`, `bmi`, an `is_goalkeeper` flag, and grouped attribute means such as `attacking_mean`, `defending_mean`, and `goalkeeping_mean`. These columns reduce repeated logic across the model, batch prediction, and serving paths.


In [ ]:
num = df.select_dtypes('number')
corr = num.corr()['overall_rating'].drop('overall_rating').sort_values(ascending=False)
print('Top correlations with overall_rating:')
print(corr.head(10).round(3))
print()
print('potential vs overall_rating corr:', round(df[['potential','overall_rating']].corr().iloc[0,1], 3))

# Engineered features (age, bmi, attribute-group means, is_goalkeeper)
eng = utils.engineer_features(utils.apply_value_semantics(df))
print()
print('engineered age range:', round(eng['age'].min(),1), '-', round(eng['age'].max(),1))
print('goalkeepers flagged:', int(eng['is_goalkeeper'].sum()), 'of', len(eng))
eng[['age','bmi','attacking_mean','defending_mean','goalkeeping_mean']].describe().round(1)
